# 🔭 Spectral Line Manager

Configure which emission lines and absorption features appear in **SpectraPyle** plots.  
Enable or disable entries, then click **Save changes** — the next `plotting()` call picks them up automatically.

> **How to use:**  run the two cells below in order, then use the checkboxes to toggle lines on/off and press **Save changes**.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

tables_dir = Path().resolve().parent / "tables"
em_path  = tables_dir / "emission_lines_vacuum_table.csv"
abs_path = tables_dir / "absorptions_table.csv"

df_em  = pd.read_csv(em_path)
df_abs = pd.read_csv(abs_path)
print(f"✓  {len(df_em)} emission lines · {len(df_abs)} absorption features loaded")

✓  254 emission lines · 45 absorption features loaded


In [2]:
def _fmt(raw):
    subs = [(r"$\alpha$", "α"), (r"$\beta$", "β"),
            (r"$\gamma$", "γ"), (r"$\delta$", "δ")]
    for s, t in subs:
        raw = raw.replace(s, t)
    return raw

_box    = widgets.Layout(width="260px", height="28px")
_scroll = widgets.Layout(
    height="420px", overflow_y="auto",
    border="1px solid #ccc", padding="4px 8px"
)

em_boxes = [
    widgets.Checkbox(
        value=bool(row["show"]),
        description=f"{_fmt(row['formatted_ion_simple'])}  {int(row['wavelength_AA'])} Å",
        style={"description_width": "initial"},
        layout=_box,
    )
    for _, row in df_em.iterrows()
]

abs_boxes = [
    widgets.Checkbox(
        value=bool(row.get("Show", 1)),
        description=f"{row['Index name']}  {int(row['Centre'])} Å",
        style={"description_width": "initial"},
        layout=_box,
    )
    for _, row in df_abs.iterrows()
]

def _ctrl(boxes):
    btn_all  = widgets.Button(description="Select all",   button_style="info",
                              layout=widgets.Layout(width="105px"))
    btn_none = widgets.Button(description="Deselect all", button_style="warning",
                              layout=widgets.Layout(width="105px"))
    btn_all.on_click(lambda  e: [setattr(b, "value", True)  for b in boxes])
    btn_none.on_click(lambda e: [setattr(b, "value", False) for b in boxes])
    return widgets.HBox([btn_all, btn_none],
                        layout=widgets.Layout(margin="4px 0 8px 0"))

def _panel(title, boxes):
    return widgets.VBox([
        widgets.HTML(f"<b style='font-size:13px; color:#333'>{title}</b>"),
        _ctrl(boxes),
        widgets.VBox(boxes, layout=_scroll),
    ], layout=widgets.Layout(margin="0 32px 0 0"))

save_btn = widgets.Button(
    description="💾  Save changes", button_style="success",
    layout=widgets.Layout(width="185px", height="38px", margin="16px 0 0 0"),
)
status = widgets.Output()

def _save(e):
    status.clear_output(wait=True)
    with status:
        df_em["show"]  = [int(b.value) for b in em_boxes]
        df_abs["Show"] = [int(b.value) for b in abs_boxes]
        df_em.to_csv(em_path, index=False)
        df_abs.to_csv(abs_path, index=False)
        n_em  = int(df_em["show"].sum())
        n_abs = int(df_abs["Show"].sum())
        print(f"✅  Saved — {n_em} emission lines active, {n_abs} absorption features active.")

save_btn.on_click(_save)

display(widgets.HBox([
    _panel("Emission Lines", em_boxes),
    _panel("Absorption Features", abs_boxes),
]))
display(widgets.VBox([save_btn, status]))